In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split

# Hugging Face Libraries
from datasets import Dataset
from transformers import DistilBertTokenizerFast

# PyTorch Libraries
import torch
from torch.utils.data import DataLoader

# Optional: Display plots inline
%matplotlib inline


/home/rebal/anaconda3/envs/cloud-computing-project/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the Balanced Dataset

Load the previously saved balanced dataset (`balanced_dataset.csv`) into a pandas DataFrame.


In [2]:
# Define the path to the balanced dataset
balanced_path = os.path.join('..', 'data', 'processed', 'balanced_dataset.csv')

# Load the balanced dataset
df_balanced = pd.read_csv(balanced_path)

# Display the first few rows
df_balanced.head()


,label,text,dataset
0,neutral,Pictures \n\nI have many pictures but I need t...,jigsaw
1,offensive,"RT @DamierGenesis: ""had all the bitches on MyS...",davidson
2,offensive,Should I tell this hoe who she look like cdfuuu,davidson
3,neutral,"You're way too nice, it gets me uncomfortable....",jigsaw
4,neutral,2D Articles \n\n2d games are an important part...,jigsaw


## 1.2 Label mapping 

Map neutral offensive and hatespeech to 0,1,2 for Distilbert.


In [3]:
label_mapping = {
    'neutral': 0,
    'offensive': 1,
    'hate_speech': 2
}

df_balanced['label'] = df_balanced['label'].map(label_mapping)

df_balanced['label'].value_counts()


label
0    31200
1    24000
2     4800
Name: count, dtype: int64

## 2. Data Split

Split data into respective train,test and val

In [4]:
# Define features and target
X = df_balanced['text'].values
y = df_balanced['label'].values

# First, split into training and temp sets (70% train, 30% temp)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

# Then, split temp into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("Training Set Size:", len(X_train))
print("Validation Set Size:", len(X_val))
print("Test Set Size:", len(X_test))

def print_class_distribution(y, split_name):
    print(f"\n{split_name} Set Label Distribution:")
    print(pd.Series(y).value_counts())

print_class_distribution(y_train, "Training")
print_class_distribution(y_val, "Validation")
print_class_distribution(y_test, "Test")


Training Set Size: 42000
Validation Set Size: 9000
Test Set Size: 9000

Training Set Label Distribution:
0    21840
1    16800
2     3360
Name: count, dtype: int64

Validation Set Label Distribution:
0    4680
1    3600
2     720
Name: count, dtype: int64

Test Set Label Distribution:
0    4680
1    3600
2     720
Name: count, dtype: int64


## 3. Tokenize

We'll use the `DistilBertTokenizerFast` from Hugging Face's `transformers` library to tokenize the text data. 


In [5]:
# Initialize the DistilBERT tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Display tokenizer information
print("Tokenizer initialized successfully!")


Tokenizer initialized successfully!


We'll tokenize the text data using the DistilBERT tokenizer.

### **Tokenization Parameters:**

- **Padding:** Pad sequences to the maximum length in the batch.
- **Truncation:** Truncate sequences longer than the maximum allowed length (512 tokens for DistilBERT).
- **Return Tensors:** Return PyTorch tensors.


In [6]:
# Define a function for batch-wise tokenization
def batch_tokenize_data(texts, labels, tokenizer, batch_size=1000, max_length=512):
    all_input_ids = []
    all_attention_masks = []
    all_labels = []
    
    total_samples = len(texts)
    print(f"Starting tokenization of {total_samples} samples...")
    
    for i in range(0, total_samples, batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_labels = labels[i:i + batch_size]
        
        # Tokenize the batch
        encodings = tokenizer(
            list(batch_texts),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        
        all_input_ids.append(encodings['input_ids'])
        all_attention_masks.append(encodings['attention_mask'])
        all_labels.append(torch.tensor(batch_labels))
        
        if (i // batch_size) % 10 == 0:
            print(f"Tokenized {i} / {total_samples} samples...")
    
    # Concatenate all batches
    input_ids = torch.cat(all_input_ids, dim=0)
    attention_masks = torch.cat(all_attention_masks, dim=0)
    labels = torch.cat(all_labels, dim=0)
    
    print("Tokenization completed.")
    
    return {'input_ids': input_ids, 'attention_mask': attention_masks}, labels

# Tokenize  data
train_encodings, train_labels = batch_tokenize_data(X_train, y_train, tokenizer, batch_size=1000)
val_encodings, val_labels = batch_tokenize_data(X_val, y_val, tokenizer, batch_size=1000)
test_encodings, test_labels = batch_tokenize_data(X_test, y_test, tokenizer, batch_size=1000)

print("\nSample Tokenized Input IDs:", train_encodings["input_ids"][0])
print("Sample Attention Mask:", train_encodings["attention_mask"][0])



Starting tokenization of 42000 samples...
Tokenized 0 / 42000 samples...
Tokenized 10000 / 42000 samples...
Tokenized 20000 / 42000 samples...
Tokenized 30000 / 42000 samples...
Tokenized 40000 / 42000 samples...
Tokenization completed.
Starting tokenization of 9000 samples...
Tokenized 0 / 9000 samples...
Tokenization completed.
Starting tokenization of 9000 samples...
Tokenized 0 / 9000 samples...
Tokenization completed.

Sample Tokenized Input IDs: tensor([  101,  8038,  1005,  2222,  9152, 13327,  2015,  2031,  1037,  2204,
        15060,   102,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0

## 4. Create Custom PyTorch Datasets

We'll define a custom `Dataset` class to handle the tokenized data and labels, making them compatible with PyTorch's `DataLoader`.


In [7]:
# Define a custom PyTorch Dataset
class HateSpeechDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: self.encodings[key][idx] for key in self.encodings}
        item['labels'] = self.labels[idx]
        return item

# Create Dataset objects
train_dataset = HateSpeechDataset(train_encodings, train_labels)
val_dataset = HateSpeechDataset(val_encodings, val_labels)
test_dataset = HateSpeechDataset(test_encodings, test_labels)

# Verify the datasets
print(f"Training Dataset Size: {len(train_dataset)}")
print(f"Validation Dataset Size: {len(val_dataset)}")
print(f"Test Dataset Size: {len(test_dataset)}")


Training Dataset Size: 42000
Validation Dataset Size: 9000
Test Dataset Size: 9000


## 5. Create DataLoaders

PyTorch's `DataLoader` will handle batching, shuffling, and efficient data loading during training and evaluation.


In [8]:
# Define batch size
batch_size = 32

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Verify DataLoader by inspecting the first batch
for batch in train_loader:
    print("Batch Keys:", batch.keys())
    print("Input IDs Shape:", batch["input_ids"].shape)
    print("Attention Mask Shape:", batch["attention_mask"].shape)
    print("Labels Shape:", batch["labels"].shape)
    break


Batch Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs Shape: torch.Size([32, 512])
Attention Mask Shape: torch.Size([32, 512])
Labels Shape: torch.Size([32])


## 6. Save Tokenized Data

In [9]:
# Define directory to save tokenized data
save_dir = os.path.join('..', 'data', 'tokenized')
os.makedirs(save_dir, exist_ok=True)

# Save tokenized datasets
torch.save((train_encodings, train_labels), os.path.join(save_dir, 'train.pt'))
torch.save((val_encodings, val_labels), os.path.join(save_dir, 'val.pt'))
torch.save((test_encodings, test_labels), os.path.join(save_dir, 'test.pt'))

print(f"Tokenized datasets saved successfully in '{save_dir}'.")


Tokenized datasets saved successfully in '../data/data_processed/tokenized'.


## 7. Verify Data Preparation

Ensure that the tokenization and dataset creation have been performed correctly by decoding a sample input and checking its label.


In [10]:
# Function to decode input IDs back to text (for verification)
def decode_batch(input_ids):
    return tokenizer.batch_decode(input_ids, skip_special_tokens=True)

# Load one sample from the training dataset
sample = train_dataset[0]

# Decode the input_ids back to text
decoded_text = decode_batch([sample['input_ids']])
print("Decoded Text:", decoded_text[0])

# Print the label
label_reverse_mapping = {0: 'neutral', 1: 'offensive', 2: 'hate_speech'}
print("Label:", label_reverse_mapping[sample['labels'].item()])


Decoded Text: ya ' ll niggers have a good thanksgiving
Label: offensive
